In [31]:
import pandas as pd
import numpy as np
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import random
import subprocess
import warnings
import cv2
from PIL import Image
import torch
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

All libraries imported successfully!
PyTorch version: 2.10.0.dev20251018+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
GPU Memory: 17.09 GB


In [32]:
BASE_DIR = r"C:\Users\AdminPC\Downloads\Week6"
DATA_DIR = os.path.join(BASE_DIR, "data_set")
TRAIN_CSV = os.path.join(DATA_DIR, "Bounding_boxes", "train_labels.csv")
TEST_CSV = os.path.join(DATA_DIR, "Bounding_boxes", "test_labels.csv")
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "images", "train")
TEST_IMAGES_DIR = os.path.join(DATA_DIR, "images", "test")
TRAIN_LABELS_DIR = os.path.join(DATA_DIR, "labels", "train")
TEST_LABELS_DIR = os.path.join(DATA_DIR, "labels", "test")
YOLO_DIR = os.path.join(BASE_DIR, "yolov5")
PROJECT_DIR = os.path.join(BASE_DIR, "graffiti_project")
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

# Training Configuration - OPTIMIZED FOR RTX 5080 (16GB VRAM)
# Matching PDF Requirements: 400 train images, 40 test images
IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 50  

TRAIN_IMAGES_PER_ITERATION = 400
VAL_IMAGES_PER_ITERATION = 0
TEST_IMAGES_PER_ITERATION = 40
TARGET_IOU_PERCENTAGE = 80
TARGET_IOU_THRESHOLD = 0.9

print("\nYOLO v5 GRAFFITI DETECTION - IMPROVED CONFIGURATION")
print(f"Base Directory: {BASE_DIR}")
print(f"Project Directory: {PROJECT_DIR}")
print(f"YOLOv5 Directory: {YOLO_DIR}")
print("\n*** HIGH-IMPACT IMPROVEMENTS ENABLED ***")
print(f"  Epochs per iteration: {EPOCHS}")
print(f"  Image size: {IMG_SIZE}px")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Target: {TARGET_IOU_PERCENTAGE}% with IoU > {TARGET_IOU_THRESHOLD}")
print("\nConfiguration (PDF Requirements):")
print(f"  Model: YOLOv5-Medium (yolov5m.pt)")
print(f"  Train images per iteration: {TRAIN_IMAGES_PER_ITERATION}")
print(f"  Test images per iteration: {TEST_IMAGES_PER_ITERATION}")


YOLO v5 GRAFFITI DETECTION - IMPROVED CONFIGURATION
Base Directory: C:\Users\AdminPC\Downloads\Week6
Project Directory: C:\Users\AdminPC\Downloads\Week6\graffiti_project
YOLOv5 Directory: C:\Users\AdminPC\Downloads\Week6\yolov5

*** HIGH-IMPACT IMPROVEMENTS ENABLED ***
  Epochs per iteration: 50
  Image size: 640px
  Batch size: 16
  Target: 80% with IoU > 0.9

Configuration (PDF Requirements):
  Model: YOLOv5-Medium (yolov5m.pt)
  Train images per iteration: 400
  Test images per iteration: 40


In [33]:
all_files = os.listdir(TRAIN_IMAGES_DIR)
# Use .lower() to catch both .jpg and .JPG, .png and .PNG
jpg_files = [f for f in all_files if f.lower().endswith('.jpg')]
png_files = [f for f in all_files if f.lower().endswith('.png')]
other_files = [f for f in all_files if not f.lower().endswith(('.jpg', '.png', '.txt'))]

print(f"Total files: {len(all_files)}")
print(f"JPG files: {len(jpg_files)}")
print(f"PNG files: {len(png_files)}")
print(f"Other files: {len(other_files)}")
print(f"Total images: {len(jpg_files) + len(png_files)}")

if other_files:
    print(f"\nFound {len(other_files)} other files (will be ignored):")
    for f in other_files[:5]:
        print(f"  - {f}")
    if len(other_files) > 5:
        print(f"  ... and {len(other_files) - 5} more")

Total files: 813
JPG files: 813
PNG files: 0
Other files: 0
Total images: 813


In [34]:
def convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_width, img_height):
    """Convert corner format to YOLO format (normalized center coordinates)"""
    x_center = (xmin + xmax) / 2.0 / img_width
    y_center = (ymin + ymax) / 2.0 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    return x_center, y_center, width, height


def convert_csv_to_yolo():
    """Convert CSV annotation files to YOLO format - FIXED for case-insensitive extensions"""
    print("\nSTEP 1: CONVERTING ANNOTATIONS TO YOLO FORMAT")
    
    if os.path.exists(TRAIN_LABELS_DIR) and len(os.listdir(TRAIN_LABELS_DIR)) > 0:
        print("Labels already converted, skipping...")
        return
    
    os.makedirs(TRAIN_LABELS_DIR, exist_ok=True)
    os.makedirs(TEST_LABELS_DIR, exist_ok=True)
    
    for csv_path, output_dir, name in [(TRAIN_CSV, TRAIN_LABELS_DIR, "Train"),
                                        (TEST_CSV, TEST_LABELS_DIR, "Test")]:
        print(f"\nConverting {name} labels...")
        df = pd.read_csv(csv_path)
        grouped = df.groupby('filename')
        
        converted = 0
        for filename, group in grouped:
            annotations = []
            for _, row in group.iterrows():
                x_c, y_c, w, h = convert_bbox_to_yolo(
                    row['xmin'], row['ymin'], row['xmax'], row['ymax'],
                    row['width'], row['height']
                )
                annotations.append([0, x_c, y_c, w, h])
            
            # FIX: Handle both .jpg/.JPG and .png/.PNG case-insensitively
            label_filename = filename
            for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
                if label_filename.endswith(ext):
                    label_filename = label_filename[:-len(ext)] + '.txt'
                    break
            
            label_file = os.path.join(output_dir, label_filename)
            with open(label_file, 'w') as f:
                for ann in annotations:
                    f.write(f"{ann[0]} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n")
            converted += 1
        
        print(f"Converted {converted} {name} labels")
    
    print("\nConversion complete!")


def select_random_images(source_dir, num_images, used_images):
    """Select random images, allowing reuse after first complete pass"""
    all_images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.jpg', '.png'))]
    available = [img for img in all_images if img not in used_images]
    
    if len(available) < num_images:
        print(f"  Image pool exhausted - allowing reuse (completed one full pass)")
        available = all_images
        used_images.clear()
    
    if len(available) < num_images:
        print(f"  Warning: Only {len(available)} images available, requested {num_images}")
        num_images = len(available)
    
    return random.sample(available, num_images) if num_images > 0 else []


def copy_dataset(image_list, src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir):
    """Copy images and their corresponding labels - FIXED for case-insensitive extensions"""
    for img in image_list:
        # Copy image
        shutil.copy(os.path.join(src_img_dir, img), 
                   os.path.join(dst_img_dir, img))
        
        # Create label filename (handle case-insensitive extensions)
        lbl_name = img
        for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
            if lbl_name.endswith(ext):
                lbl_name = lbl_name[:-len(ext)] + '.txt'
                break
        
        src_lbl = os.path.join(src_lbl_dir, lbl_name)
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, os.path.join(dst_lbl_dir, lbl_name))
        else:
            print(f"  Warning: Label not found for {img}")

print("Helper functions defined")

# NOW CALL THE CONVERSION FUNCTION
convert_csv_to_yolo()

Helper functions defined

STEP 1: CONVERTING ANNOTATIONS TO YOLO FORMAT
Labels already converted, skipping...


In [35]:
# Check counts
train_imgs = len([f for f in os.listdir(TRAIN_IMAGES_DIR) if f.lower().endswith(('.jpg', '.png'))])
train_lbls = len([f for f in os.listdir(TRAIN_LABELS_DIR) if f.endswith('.txt')])
print(f"Train: {train_imgs} images, {train_lbls} labels")

Train: 813 images, 813 labels


In [36]:
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes"""
    x1_min = box1[0] - box1[2] / 2
    y1_min = box1[1] - box1[3] / 2
    x1_max = box1[0] + box1[2] / 2
    y1_max = box1[1] + box1[3] / 2
    
    x2_min = box2[0] - box2[2] / 2
    y2_min = box2[1] - box2[3] / 2
    x2_max = box2[0] + box2[2] / 2
    y2_max = box2[1] + box2[3] / 2
    
    x_inter_min = max(x1_min, x2_min)
    y_inter_min = max(y1_min, y2_min)
    x_inter_max = min(x1_max, x2_max)
    y_inter_max = min(y1_max, y2_max)
    
    if x_inter_max < x_inter_min or y_inter_max < y_inter_min:
        return 0.0
    
    intersection = (x_inter_max - x_inter_min) * (y_inter_max - y_inter_min)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def calculate_max_iou_for_image(gt_boxes, pred_boxes):
    """Calculate maximum IoU for image with multiple boxes"""
    if not gt_boxes or not pred_boxes:
        return 0.0
    
    max_iou = 0.0
    for gt_box in gt_boxes:
        for pred_box in pred_boxes:
            iou = calculate_iou(pred_box[:4], gt_box)
            max_iou = max(max_iou, iou)
    
    return max_iou

print("IoU calculation functions defined")

IoU calculation functions defined


In [37]:
def train_iteration(iteration_num, prev_model=None):
    """Train one iteration with RTX 5080 optimized configuration"""
    print(f"\nITERATION {iteration_num}: TRAINING")

    iter_dir = os.path.join(PROJECT_DIR, "iterations", f"iteration_{iteration_num}")
    os.makedirs(iter_dir, exist_ok=True)
    
    total_needed = TRAIN_IMAGES_PER_ITERATION + VAL_IMAGES_PER_ITERATION
    train_imgs = select_random_images(TRAIN_IMAGES_DIR, total_needed, used_train_images)
    
    if len(train_imgs) < total_needed:
        print(f"Cannot train - only {len(train_imgs)} images available (need {total_needed})")
        return None
    
    used_train_images.update(train_imgs)
    
    random.shuffle(train_imgs)
    
    # PDF Requirement: Use all 400 images for training
    # If VAL_IMAGES_PER_ITERATION = 0, use all for training
    # If VAL_IMAGES_PER_ITERATION > 0, split accordingly
    if VAL_IMAGES_PER_ITERATION > 0:
        train_split = train_imgs[:TRAIN_IMAGES_PER_ITERATION]
        val_split = train_imgs[TRAIN_IMAGES_PER_ITERATION:total_needed]
    else:
        train_split = train_imgs
        val_split = train_imgs[:40]  # Use small subset for validation (YOLOv5 requirement)
    
    print(f"Selected: {len(train_split)} train, {len(val_split)} validation images")
    print(f"Total images for training: {len(train_imgs)}")
    
    # Prepare dataset directories
    for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
        path = os.path.join(DATASET_DIR, folder)
        os.makedirs(path, exist_ok=True)
        for f in os.listdir(path):
            os.remove(os.path.join(path, f))
    
    print("Copying dataset files...")
    copy_dataset(train_split, TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR,
                os.path.join(DATASET_DIR, "images/train"),
                os.path.join(DATASET_DIR, "labels/train"))
    
    copy_dataset(val_split, TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR,
                os.path.join(DATASET_DIR, "images/val"),
                os.path.join(DATASET_DIR, "labels/val"))
    
    # Verify copied files
    train_img_count = len(os.listdir(os.path.join(DATASET_DIR, "images/train")))
    train_lbl_count = len(os.listdir(os.path.join(DATASET_DIR, "labels/train")))
    val_img_count = len(os.listdir(os.path.join(DATASET_DIR, "images/val")))
    val_lbl_count = len(os.listdir(os.path.join(DATASET_DIR, "labels/val")))
    
    print(f"Dataset prepared:")
    print(f"  Train: {train_img_count} images, {train_lbl_count} labels")
    print(f"  Val:   {val_img_count} images, {val_lbl_count} labels")
    
    if train_lbl_count == 0 or val_lbl_count == 0:
        print("ERROR: No labels found! Check label conversion.")
        return None
    
    # Create YAML with augmentation
    yaml_path = os.path.join(DATASET_DIR, "data.yaml")
    with open(yaml_path, 'w') as f:
        f.write(f"""path: {DATASET_DIR}
train: images/train
val: images/val
nc: 1
names: ['graffiti']

# Data Augmentation
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 10.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 1.0
mixup: 0.0
""")
    
    print(f"\nYAML created: {yaml_path}")
    
    # Use larger model
    weights = prev_model if prev_model else "yolov5m.pt"
    
    # Optimized training command for RTX 5080
    train_cmd = [
        'python', 'train.py',
        '--img', str(IMG_SIZE),
        '--batch', str(BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--data', yaml_path,
        '--weights', weights,
        '--project', iter_dir,
        '--name', 'train',
        '--device', '0',
        '--workers', '16',  # Increased workers for faster data loading
        '--cache', 'ram',  # Cache in RAM (you have plenty!)
        '--exist-ok'
    ]
    
    print(f"\nTraining with: {os.path.basename(weights)}")
    print(f"Config: {EPOCHS} epochs, batch={BATCH_SIZE}, img={IMG_SIZE}")
    print(f"Workers: 16 (optimized for RTX 5080)")
    print(f"Cache: RAM (fast loading)")
    print(f"Data augmentation: Enabled")
    print("\nTraining started...\n")
    
    # Run training
    process = subprocess.Popen(
        train_cmd,
        cwd=YOLO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    
    for line in process.stdout:
        line = line.strip()
        if line and 'Comet' not in line:
            if any(keyword in line for keyword in ['Epoch', 'all', 'mAP', 'Results', 
                                                    'Optimizer', 'Starting', 'error', 
                                                    'Error', 'WARNING', 'labels']):
                print(line)
    
    process.wait()
    
    # Check training success
    best_src = os.path.join(iter_dir, "train", "weights", "best.pt")
    best_dst = os.path.join(iter_dir, "best.pt")
    
    if os.path.exists(best_src):
        shutil.copy(best_src, best_dst)
        print(f"\nModel saved: {best_dst}")
        
        # Check if model actually learned something
        results_file = os.path.join(iter_dir, "train", "results.csv")
        if os.path.exists(results_file):
            df = pd.read_csv(results_file)
            df.columns = df.columns.str.strip()  # Remove whitespace from column names
            
            # Try different possible column names for mAP50
            map_col = None
            for col in ['metrics/mAP50(B)', 'metrics/mAP_0.5', 'mAP_0.5', 'mAP@0.5']:
                if col in df.columns:
                    map_col = col
                    break
            
            if map_col:
                final_map = df[map_col].iloc[-1]
                print(f"Final mAP50: {final_map:.3f}")
                if final_map < 0.01:
                    print("WARNING: Model barely learned anything (mAP < 0.01)")
            else:
                print(f"Available columns: {list(df.columns)}")
                print("Could not find mAP50 column, but training completed successfully")
        
        return best_dst
    else:
        print("Training failed - model not found!")
        return None

print("Training function defined")

Training function defined


In [38]:
def test_iteration(iteration_num, model_path):
    """Test one iteration with 40 test images"""
    print(f"\nITERATION {iteration_num}: TESTING")

    iter_dir = os.path.join(PROJECT_DIR, "iterations", f"iteration_{iteration_num}")
    
    test_imgs = select_random_images(TEST_IMAGES_DIR, TEST_IMAGES_PER_ITERATION, used_test_images)
    
    if len(test_imgs) < TEST_IMAGES_PER_ITERATION:
        print(f"Only {len(test_imgs)} test images available (requested {TEST_IMAGES_PER_ITERATION})")
    
    used_test_images.update(test_imgs)
    
    print(f"Testing on {len(test_imgs)} images...")
    
    # Create temp directory for test images
    temp_dir = os.path.join(PROJECT_DIR, "temp_test")
    os.makedirs(temp_dir, exist_ok=True)
    for img in test_imgs:
        shutil.copy(os.path.join(TEST_IMAGES_DIR, img), temp_dir)
    
    # Run detection
    detect_dir = os.path.join(iter_dir, "detect")
    detect_cmd = [
        'python', 'detect.py',
        '--weights', model_path,
        '--source', temp_dir,
        '--project', detect_dir,
        '--name', 'test',
        '--save-txt',
        '--save-conf',
        '--conf', '0.25',
        '--device', '0',
        '--exist-ok'
    ]
    
    print("Running detection...")
    subprocess.run(detect_cmd, cwd=YOLO_DIR, capture_output=True)
    
    # Calculate IoU for each image
    results = []
    detect_labels = os.path.join(detect_dir, "test", "labels")
    
    for img in test_imgs:
        gt_file = os.path.join(TEST_LABELS_DIR, img.replace('.jpg', '.txt').replace('.png', '.txt'))
        pred_file = os.path.join(detect_labels, img.replace('.jpg', '.txt').replace('.png', '.txt'))
        
        # Load ground truth boxes
        gt_boxes = []
        if os.path.exists(gt_file):
            with open(gt_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        gt_boxes.append([float(parts[i]) for i in range(1, 5)])
        
        # Load predicted boxes
        pred_boxes = []
        max_conf = 0.0
        if os.path.exists(pred_file):
            with open(pred_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 6:
                        pred_box = [float(parts[i]) for i in range(1, 5)]
                        conf = float(parts[5])
                        pred_boxes.append(pred_box + [conf])
                        max_conf = max(max_conf, conf)
        
        # Calculate IoU
        if gt_boxes and pred_boxes:
            iou = calculate_max_iou_for_image(gt_boxes, pred_boxes)
        else:
            iou = 0.0
        
        results.append({
            'image_name': img,
            'confidence': max_conf,
            'iou': iou
        })
    
    # Save CSV
    df = pd.DataFrame(results)
    csv_path = os.path.join(iter_dir, "results.csv")
    df.to_csv(csv_path, index=False)
    print(f"Results CSV saved: {csv_path}")
    
    # Save 2 best sample images
    samples_dir = os.path.join(iter_dir, "samples")
    os.makedirs(samples_dir, exist_ok=True)
    
    best_2 = df.nlargest(2, 'iou')
    detect_imgs = os.path.join(detect_dir, "test")
    
    saved_count = 0
    print("\nSaving sample images with bounding boxes:")
    for idx, row in best_2.iterrows():
        src = os.path.join(detect_imgs, row['image_name'])
        dst = os.path.join(samples_dir, 
                          f"sample{saved_count+1}_iou{row['iou']:.3f}_conf{row['confidence']:.3f}.jpg")
        if os.path.exists(src):
            shutil.copy(src, dst)
            saved_count += 1
            print(f"  Sample {saved_count}: {row['image_name']} (IoU: {row['iou']:.3f}, Conf: {row['confidence']:.3f})")
    
    if saved_count < 2:
        print(f"  Warning: Only saved {saved_count} sample images")
    
    # Clean up
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    # Calculate statistics
    avg_iou = df['iou'].mean()
    avg_conf = df['confidence'].mean()
    high_iou_count = (df['iou'] > TARGET_IOU_THRESHOLD).sum()
    percentage = (high_iou_count / len(df)) * 100
    
    print("\nRESULTS SUMMARY:")
    print(f"  Average IoU:        {avg_iou:.3f}")
    print(f"  Average Confidence: {avg_conf:.3f}")
    print(f"  High IoU (>{TARGET_IOU_THRESHOLD}):   {high_iou_count}/{len(df)} images ({percentage:.1f}%)")
    print(f"  Target:             {TARGET_IOU_PERCENTAGE}%")
    print(f"  Results CSV:        {csv_path}")
    print(f"  Sample images:      {samples_dir}")
    print(f"  Best model:         {model_path}")
    
    target_achieved = percentage >= TARGET_IOU_PERCENTAGE
    
    if target_achieved:
        print(f"\nTARGET ACHIEVED! {percentage:.1f}% >= {TARGET_IOU_PERCENTAGE}%")
    else:
        print(f"\nTarget not reached. Gap: {TARGET_IOU_PERCENTAGE - percentage:.1f}%")
    
    stats = {
        'iteration': iteration_num,
        'avg_iou': avg_iou,
        'avg_conf': avg_conf,
        'high_iou_percentage': percentage,
        'high_iou_count': high_iou_count,
        'total_images': len(df),
        'csv_path': csv_path,
        'model_path': model_path,
        'samples_dir': samples_dir
    }
    
    return target_achieved, stats

print("Testing function defined")

Testing function defined


In [39]:
used_train_images = set()
used_test_images = set()

iteration = 1
target_achieved = False
prev_model = None
max_iterations = 4
all_stats = []

print("\nSTARTING ITERATIVE TRAINING AND TESTING")
print(f"Goal: {TARGET_IOU_PERCENTAGE}% of images with IoU > {TARGET_IOU_THRESHOLD}")
print(f"Max iterations: {max_iterations}")
print(f"Model: YOLOv5-Medium")
print(f"Image size: {IMG_SIZE}px")
print(f"Batch size: {BATCH_SIZE} (RTX 5080 optimized)")
print(f"Epochs per iteration: {EPOCHS}")

while not target_achieved and iteration <= max_iterations:
    print(f"\n{'='*60}")
    print(f"### ITERATION {iteration} START ###")
    print(f"{'='*60}")
    
    # Check available data
    available_train = len([f for f in os.listdir(TRAIN_IMAGES_DIR) 
                          if f.lower().endswith(('.jpg', '.png')) and f not in used_train_images])
    available_test = len([f for f in os.listdir(TEST_IMAGES_DIR) 
                         if f.lower().endswith(('.jpg', '.png')) and f not in used_test_images])
    
    train_needed = TRAIN_IMAGES_PER_ITERATION + VAL_IMAGES_PER_ITERATION
    test_needed = TEST_IMAGES_PER_ITERATION
    
    print(f"\nData availability:")
    print(f"  Available unused: {available_train} train, {available_test} test")
    print(f"  Needed: {train_needed} train, {test_needed} test")
    
    # TRAIN
    best_model = train_iteration(iteration, prev_model)
    
    if not best_model:
        print(f"\nTraining failed at iteration {iteration}")
        break
    
    # TEST
    target_achieved, stats = test_iteration(iteration, best_model)
    
    if stats:
        all_stats.append(stats)
    
    if target_achieved:
        print(f"\n{'='*60}")
        print(f"### SUCCESS! TARGET ACHIEVED IN ITERATION {iteration} ###")
        print(f"{'='*60}")
        print(f"\nFinal model: {best_model}")
        print(f"High IoU percentage: {stats['high_iou_percentage']:.1f}%")
        break
    
    prev_model = best_model
    iteration += 1

if not target_achieved and iteration > max_iterations:
    print(f"\nMaximum iterations ({max_iterations}) reached without achieving target")


STARTING ITERATIVE TRAINING AND TESTING
Goal: 80% of images with IoU > 0.9
Max iterations: 4
Model: YOLOv5-Medium
Image size: 640px
Batch size: 16 (RTX 5080 optimized)
Epochs per iteration: 50

### ITERATION 1 START ###

Data availability:
  Available unused: 813 train, 209 test
  Needed: 400 train, 40 test

ITERATION 1: TRAINING
Selected: 400 train, 40 validation images
Total images for training: 400
Copying dataset files...
Dataset prepared:
  Train: 400 images, 400 labels
  Val:   40 images, 40 labels

YAML created: C:\Users\AdminPC\Downloads\Week6\graffiti_project\dataset\data.yaml

Training with: yolov5m.pt
Config: 50 epochs, batch=16, img=640
Workers: 16 (optimized for RTX 5080)
Cache: RAM (fast loading)
Data augmentation: Enabled

Training started...

train: Scanning C:\Users\AdminPC\Downloads\Week6\graffiti_project\dataset\labels\train...:   0%|          | 0/400 [00:00<?, ?it/s]C:\Users\AdminPC\.conda\envs\yolo_rtx5080\lib\site-packages\requests\__init__.py:86: RequestsDepende

In [40]:
print("\n" + "="*60)
print("FINAL TRAINING & TESTING SUMMARY")
print("="*60)
print(f"Total iterations completed: {len(all_stats)}")
print(f"Target achieved: {'YES' if target_achieved else 'NO'}")

for stat in all_stats:
    print(f"\nIteration {stat['iteration']}:")
    print(f"  Average IoU:        {stat['avg_iou']:.3f}")
    print(f"  Average Confidence: {stat['avg_conf']:.3f}")
    print(f"  High IoU count:     {stat['high_iou_count']}/{stat['total_images']}")
    print(f"  High IoU percentage: {stat['high_iou_percentage']:.1f}%")
    print(f"  Results CSV:        {stat['csv_path']}")
    print(f"  Samples:            {stat['samples_dir']}")
    print(f"  Model:              {stat['model_path']}")

print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)


FINAL TRAINING & TESTING SUMMARY
Total iterations completed: 4
Target achieved: NO

Iteration 1:
  Average IoU:        0.759
  Average Confidence: 0.772
  High IoU count:     22/40
  High IoU percentage: 55.0%
  Results CSV:        C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_1\results.csv
  Samples:            C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_1\samples
  Model:              C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_1\best.pt

Iteration 2:
  Average IoU:        0.710
  Average Confidence: 0.776
  High IoU count:     22/40
  High IoU percentage: 55.0%
  Results CSV:        C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_2\results.csv
  Samples:            C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_2\samples
  Model:              C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_2\best.pt

Iteration 3:
  Average IoU:        

In [44]:
def detect_video(video_path, model_path, output_dir):
    """Detect graffiti in video using trained model - FIXED"""
    video_name = os.path.basename(video_path)
    video_base = video_name.rsplit('.', 1)[0]
    video_ext = video_name.rsplit('.', 1)[-1]
    
    print(f"\nProcessing: {video_name}")
    
    # Create a temporary detection directory
    temp_detect_dir = os.path.join(PROJECT_DIR, "temp_video_detect")
    
    detect_cmd = [
        'python', 'detect.py',
        '--weights', model_path,
        '--source', video_path,
        '--project', temp_detect_dir,
        '--name', 'result',
        '--conf', '0.15',
        '--device', '0',
        '--exist-ok'
    ]
    
    print("Running detection...")
    result = subprocess.run(detect_cmd, cwd=YOLO_DIR, capture_output=True, text=True)
    
    if result.returncode == 0:
        # YOLOv5 saves video in: temp_detect_dir/result/video_name
        source_video = os.path.join(temp_detect_dir, "result", video_name)
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        
        # Destination with "detected_" prefix
        final_video = os.path.join(output_dir, f"detected_{video_name}")
        
        if os.path.exists(source_video):
            shutil.copy(source_video, final_video)
            print(f"✓ Video processed successfully!")
            print(f"  Saved as: detected_{video_name}")
            
            # Clean up temp directory
            shutil.rmtree(temp_detect_dir, ignore_errors=True)
            return True
        else:
            print(f"✗ Processed video not found at: {source_video}")
            print(f"  Check: {temp_detect_dir}")
            return False
    else:
        print(f"✗ Error during detection")
        if result.stderr:
            print(result.stderr[:500])  # Print first 500 chars of error
        return False

print("\n Video detection ")


 Video detection 


In [45]:
# Process videos with the best trained model
video_dir = os.path.join(PROJECT_DIR, 'videos')
output_dir = os.path.join(PROJECT_DIR, 'video_results')
best_model = all_stats[-1]['model_path']  # Use best model from training

# Create videos directory if it doesn't exist
os.makedirs(video_dir, exist_ok=True)

print(f"\nVideo Processing Setup:")
print(f"  Videos folder: {video_dir}")
print(f"  Output folder: {output_dir}")
print(f"  Model: {best_model}")
print(f"\nPlace your 5 videos in: {video_dir}")
print("Then run the next cell to process them.")


Video Processing Setup:
  Videos folder: C:\Users\AdminPC\Downloads\Week6\graffiti_project\videos
  Output folder: C:\Users\AdminPC\Downloads\Week6\graffiti_project\video_results
  Model: C:\Users\AdminPC\Downloads\Week6\graffiti_project\iterations\iteration_4\best.pt

Place your 5 videos in: C:\Users\AdminPC\Downloads\Week6\graffiti_project\videos
Then run the next cell to process them.


In [46]:
# Process all videos in the videos folder
video_dir = os.path.join(PROJECT_DIR, 'videos')
output_dir = os.path.join(PROJECT_DIR, 'video_results')
best_model = all_stats[-1]['model_path']

# Get list of video files
video_files = [f for f in os.listdir(video_dir) 
               if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

print(f"\nFound {len(video_files)} videos to process:")
for i, video in enumerate(video_files, 1):
    print(f"  {i}. {video}")

# Process each video
print("\n" + "="*60)
print("STARTING VIDEO PROCESSING")
print("="*60)

for i, video in enumerate(video_files, 1):
    print(f"\n[{i}/{len(video_files)}] Processing: {video}")
    video_path = os.path.join(video_dir, video)
    success = detect_video(video_path, best_model, output_dir)
    
    if success:
        print(f"✓ Successfully processed {video}")
    else:
        print(f"✗ Failed to process {video}")

print("\n" + "="*60)
print("VIDEO PROCESSING COMPLETE")
print("="*60)
print(f"\nResults saved in: {output_dir}")


Found 4 videos to process:
  1. 3413463-hd_1920_1080_30fps.mp4
  2. 4543511-hd_1080_1920_25fps.mp4
  3. 854181-hd_1920_1080_25fps.mp4
  4. 9724130-hd_1440_1080_30fps.mp4

STARTING VIDEO PROCESSING

[1/4] Processing: 3413463-hd_1920_1080_30fps.mp4

Processing: 3413463-hd_1920_1080_30fps.mp4
Running detection...
✓ Video processed successfully!
  Saved as: detected_3413463-hd_1920_1080_30fps.mp4
✓ Successfully processed 3413463-hd_1920_1080_30fps.mp4

[2/4] Processing: 4543511-hd_1080_1920_25fps.mp4

Processing: 4543511-hd_1080_1920_25fps.mp4
Running detection...
✓ Video processed successfully!
  Saved as: detected_4543511-hd_1080_1920_25fps.mp4
✓ Successfully processed 4543511-hd_1080_1920_25fps.mp4

[3/4] Processing: 854181-hd_1920_1080_25fps.mp4

Processing: 854181-hd_1920_1080_25fps.mp4
Running detection...
✓ Video processed successfully!
  Saved as: detected_854181-hd_1920_1080_25fps.mp4
✓ Successfully processed 854181-hd_1920_1080_25fps.mp4

[4/4] Processing: 9724130-hd_1440_1080_3